[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [sqlite3, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlite3-deep-dive.html)

# Row Factories


## What you will be able to do

Read rows by column name with `sqlite3.Row`, turn rows into dictionaries that JSON can write, and
build rows as instances of a dataclass with a row factory of your own. Set a factory for a whole
connection or for one cursor, choose between a tuple, a `Row`, a dictionary and a dataclass for a
job, and recognize the two lookups by name that go wrong without an error: a name in another case,
and two columns with the same name.


## The idea

### The problem

Every row this guide has fetched so far came back as a tuple, and code read it by position: `row[0]`
for the name and `row[1]` for the latitude, because that is where the `SELECT` put them. Position is
a fragile way to name something. Add `id` to the front of the query's columns and every position
after it moves along by one, so code that read `row[1]` as a latitude now reads a station's name,
and a report prints it where a number belongs, with no error. A row handed to another function
carries no hint of what its positions mean, so whoever reads `row[3]` has to go and find the query.
And code written against JSON from an API reaches for `row["celsius"]` and `row.get("celsius")`,
both of which a tuple refuses.

The shape of a row is not fixed. A connection can be told to build every row as something that
answers to a column's name, and the query decides those names.

### What a row factory is

> A **row factory** is the function a cursor calls to turn each row SQLite returns into the object
> that `fetchone`, `fetchall` and a loop hand back. It is given the cursor, whose `description` names
> the result's columns, and a tuple of the row's values, and it can return anything. The default,
> `None`, returns the tuple itself. `sqlite3.Row`, which sqlite3 provides, returns a row that
> answers to a column's position and to its name, in any case. A factory is set on a connection, as
> `conn.row_factory`, for every cursor made from it afterward, or on one cursor, as
> `cursor.row_factory`, for that cursor alone.

### Why it works that way

- **A tuple is the cheapest row.** SQLite hands back values in the order of the `SELECT`, and a
  tuple keeps them in that order with nothing added. Every other kind of row is built from that
  tuple.
- **Names come from the query, not from the table.** A column's name in a row is the name in
  `description`: the column's own name, the name after `AS`, or the text of an expression such as
  `MIN(celsius)`. The query decides which names its rows answer to.
- **`sqlite3.Row` adds names without copying anything.** It keeps the row's tuple and the cursor's
  `description`, which every row of the result shares, and the Python documentation describes it as
  having minimal memory overhead and performance impact over a tuple.
- **A `Row` is a sequence that can look up names, not a dictionary.** It has `keys()` and
  `row["name"]`, but no `get`, `in` looks among its values, and `json` cannot write it. `dict(row)`
  makes a real dictionary wherever one is needed.
- **A `Row` ignores the case of a name, and a dictionary does not.** `row["coldest"]` finds a column
  the query named `Coldest`, while `dict(row)` keeps the name exactly as the query wrote it.
- **A cursor copies its factory when it is made.** Changing `conn.row_factory` leaves existing
  cursors as they were, while `conn.execute`, which makes a new cursor every time, always uses the
  connection's current factory.

### Where this shows up

Every database driver decides what a row is, and most let you choose. psycopg, in the **asyncpg and
psycopg3, Deep Dive** guide, takes a `row_factory` too, and ships `dict_row`, `namedtuple_row` and
`class_row`, which builds rows as instances of a class such as a dataclass. asyncpg returns a
`Record`, which answers to names and positions as `sqlite3.Row` does and also has `get`, so code
moved from one driver to the other trips over the difference. The `Row` in the **SQLAlchemy, Deep
Dive** guide behaves like a named tuple, with `._mapping` for lookups by name. The dictionaries this
notebook builds are the kind the **JSON in a Response** notebook in the **APIs and JSON** guide took
apart, and the dataclass factory builds on the **Dataclasses** notebook in the **Object-Oriented
Python** guide.

### What this notebook covers

- Rows as tuples, and what reading by position costs
- `sqlite3.Row`: by name, by position, `keys()` and `dict(row)`
- What a `Row` is not: no `get`, `in` looks at values, and no JSON
- A dictionary factory, built from `description`
- A dataclass factory, with a field for every column and a method of its own
- A factory for a connection, or for one cursor
- When to use a tuple, a `Row`, a dictionary or a dataclass
- A station report that hands the program objects and a client JSON
- Six errors: a tuple read by name, `get` on a `Row`, a name the query never gave, a `Row` handed to
  `json`, a name in another case that JSON loses, and two columns named `id`

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
import sqlite3

conn = sqlite3.connect(":memory:")
conn.execute("CREATE TABLE stations (name TEXT, latitude REAL)")
conn.executemany("INSERT INTO stations VALUES (?, ?)", [("Bergen", 60.39), ("Svalbard", 78.22)])
query = "SELECT name, latitude FROM stations ORDER BY latitude"

print("tuple:", conn.execute(query).fetchone())

conn.row_factory = sqlite3.Row
row = conn.execute(query).fetchone()
print("Row:  ", row["name"], row["latitude"], row[0], row.keys())
print("dict: ", dict(row))
conn.close()
```

```
tuple: ('Bergen', 60.39)
Row:   Bergen 60.39 Bergen ['name', 'latitude']
dict:  {'name': 'Bergen', 'latitude': 60.39}
```

The same query, run twice. The first time the row was a tuple, which answers only to a position.
After one assignment to `conn.row_factory`, the row answered to a column's name and still to its
position, listed its names with `keys()`, and became a dictionary with `dict`. The query did not
change at all.


## Setup

Eight imports, and the stations' year, built into the two tables the **Tables and Queries** notebook
designed.

- `sqlite3` builds the database and runs every statement
- `dataclass` and `asdict`, from `dataclasses`, define the class one factory builds, and turn an
  instance back into a dictionary
- `json` writes rows as JSON
- `re` hides the memory address in a printed row, which is different on every run
- `math`, `datetime` and `timedelta` make the same year of readings the **Why sqlite3** notebook made
- `Path` names the scratch folder and the database in it
- `shutil` removes the scratch folder at the end


In [1]:
import json
import math
import re
import shutil
import sqlite3
from dataclasses import asdict, dataclass
from datetime import datetime, timedelta
from pathlib import Path

SCRATCH = Path("scratch")
SCRATCH.mkdir(exist_ok=True)
DATABASE = SCRATCH / "stations.db"
DATABASE.unlink(missing_ok=True)
STATIONS = {"Bergen": 8.0, "Oslo": 6.5, "Svalbard": -4.5, "Tromso": 3.5}   # each station's mean for the year
LATITUDES = {"Bergen": 60.39, "Oslo": 59.91, "Svalbard": 78.22, "Tromso": 69.65, "Kirkenes": 69.73}


def year_of_readings():
    """Every hour of 2025 at the four stations, as Why sqlite3 made them, with Svalbard silent on 2 March."""
    for n in range(365 * 24):
        hour = datetime(2025, 1, 1) + timedelta(hours=n)
        season = -math.cos(2 * math.pi * (n - 400) / (365 * 24))
        day = -math.cos(2 * math.pi * (hour.hour - 3) / 24)
        for i, (station, mean) in enumerate(STATIONS.items()):
            if station == "Svalbard" and hour.strftime("%Y-%m-%d") == "2025-03-02":
                celsius = None
            else:
                wobble = ((n * 37 + i * 101) % 17 - 8) / 10
                celsius = round(mean + 9 * season + 3 * day + wobble, 1) + 0.0
            yield station, hour.strftime("%Y-%m-%dT%H:%M"), celsius


build = sqlite3.connect(DATABASE)
build.executescript("""
    CREATE TABLE stations (id INTEGER PRIMARY KEY, name TEXT NOT NULL, latitude REAL NOT NULL);
    CREATE TABLE readings (id INTEGER PRIMARY KEY, station_id INTEGER NOT NULL REFERENCES stations (id),
                           hour TEXT NOT NULL, celsius REAL);
""")
ids = {name: build.execute("INSERT INTO stations (name, latitude) VALUES (?, ?)", (name, latitude)).lastrowid
       for name, latitude in LATITUDES.items()}
build.executemany("INSERT INTO readings (station_id, hour, celsius) VALUES (?, ?, ?)",
                  ((ids[station], hour, celsius) for station, hour, celsius in year_of_readings()))
build.commit()
build.close()

print("built", DATABASE)


built scratch/stations.db


## Worked examples

### Rows as tuples, and what reading by position costs

With no row factory, a row is a tuple of values in the order the `SELECT` listed them. The same
line of code reads `row[1]` from two queries, the second with `id` added to the front of its
columns:


In [2]:
conn = sqlite3.connect(DATABASE)

row = conn.execute("SELECT name, latitude FROM stations WHERE name = ?", ("Svalbard",)).fetchone()
print(row, "latitude:", row[1])

row = conn.execute("SELECT id, name, latitude FROM stations WHERE name = ?", ("Svalbard",)).fetchone()
print(row, "latitude:", row[1])


('Svalbard', 78.22) latitude: 78.22
(3, 'Svalbard', 78.22) latitude: Svalbard


A tuple holds values and nothing about them. `row[1]` meant the latitude only because the first
query listed `latitude` second, and once `id` joined the front, the same line printed a name as a
latitude, with no error. Unpacking at once, as in `name, latitude = row`, keeps a tuple readable,
but only for code that sits right beside the query.

### sqlite3.Row

Assigning `sqlite3.Row` to `conn.row_factory` makes every row fetched afterward answer to names as
well as positions:


In [3]:
conn.row_factory = sqlite3.Row

row = conn.execute("SELECT id, name, latitude FROM stations WHERE name = ?", ("Svalbard",)).fetchone()
print("by name:    ", row["name"], row["latitude"])
print("by position:", row[0], row[-1], row[1:])
print("in any case:", row["LATITUDE"])
print("keys:       ", row.keys())
print("as a dict:  ", dict(row))

station_id, name, latitude = row
print("unpacked:   ", station_id, name, latitude)


by name:     Svalbard 78.22
by position: 3 78.22 ('Svalbard', 78.22)
in any case: 78.22
keys:        ['id', 'name', 'latitude']
as a dict:   {'id': 3, 'name': 'Svalbard', 'latitude': 78.22}
unpacked:    3 Svalbard 78.22


`row["latitude"]` finds the column wherever the query put it, so adding `id` to the front changed
nothing for code that reads by name. A `Row` is still a sequence: it answers to positions and
slices, and unpacks like the tuple it holds. `keys()` lists the names, which it takes from the
cursor's `description`, introduced in the **Connections and Cursors** notebook, and `dict(row)`
copies the names and values into an ordinary dictionary.

### What a Row is not

A `Row` looks names up, but it is not a dictionary, and three habits from dictionaries give the
wrong answer or none:


In [4]:
print("'Svalbard' in row:  ", "Svalbard" in row)
print("'name' in row:      ", "name" in row)
print("'name' in row.keys():", "name" in row.keys())

print("printed:", re.sub(r"0x[0-9a-f]+", "0x...", repr(row)))   # the address is different on every run
print("dict(row) printed:", dict(row))
print("as JSON:", json.dumps(dict(row)))


'Svalbard' in row:   True
'name' in row:       False
'name' in row.keys(): True
printed: <sqlite3.Row object at 0x...>
dict(row) printed: {'id': 3, 'name': 'Svalbard', 'latitude': 78.22}
as JSON: {"id": 3, "name": "Svalbard", "latitude": 78.22}


`in` looks among a row's values, as it does for a tuple, so a column's name is found in `keys()`,
not in the row. Printing a row shows only what it is and where it lives in memory, not what it
holds. And `json` cannot write a `Row` at all. `dict(row)` answers all three, and the `get` that a
dictionary has, which a `Row` lacks, is one of the Common errors.

### A dictionary factory, built from description

A row factory is any function that takes the cursor and the tuple of values. This one pairs every
value with its column's name from `description`, as the Python documentation's own recipe does, and
returns a plain dictionary. The names are the ones the query chose with `AS`:


In [5]:
def dict_factory(cursor, row):
    """A row as a dictionary, keyed by the names the query gave its columns."""
    return {column[0]: value for column, value in zip(cursor.description, row)}


SUMMARY = """
    SELECT s.name AS station, COUNT(r.celsius) AS readings, MIN(r.celsius) AS coldest, MAX(r.celsius) AS warmest
    FROM stations AS s LEFT JOIN readings AS r ON r.station_id = s.id
    GROUP BY s.id
    ORDER BY s.latitude DESC
"""

conn.row_factory = dict_factory
summaries = conn.execute(SUMMARY).fetchall()

print(summaries[0])
print(summaries[1].get("coldest"), summaries[1].get("median", "no median in this query"))
print(json.dumps(summaries[:2], indent=2))


{'station': 'Svalbard', 'readings': 8736, 'coldest': -17.3, 'warmest': 8.3}
None no median in this query
[
  {
    "station": "Svalbard",
    "readings": 8736,
    "coldest": -17.3,
    "warmest": 8.3
  },
  {
    "station": "Kirkenes",
    "readings": 0,
    "coldest": null,
    "warmest": null
  }
]


Every row is a dictionary in its own right: `get` works, with a default for a key that is missing,
keys can be added, and `json.dumps` writes a list of them as it stands, with `None` as `null`, as
for Kirkenes. The keys keep the case the query gave them, and every row's dictionary builds a table
of names of its own, which costs more memory than a `Row` over a large result.

### A dataclass factory

When a row stands for a thing in the program, a class can say what that thing is. This factory
builds the same dictionary of names and values and hands it to a dataclass as keyword arguments, so
every column the query returns has to match a field by name. `StationSummary` adds what neither a
`Row` nor a dictionary has, a method of its own:


In [6]:
@dataclass
class StationSummary:
    station: str
    readings: int
    coldest: float | None
    warmest: float | None

    @property
    def spread(self):
        """Degrees between the warmest and coldest readings, or None for a station with no readings."""
        if self.coldest is None:
            return None
        return round(self.warmest - self.coldest, 1)


def summary_factory(cursor, row):
    """A row as a StationSummary, with every column matched to the field of the same name."""
    return StationSummary(**{column[0]: value for column, value in zip(cursor.description, row)})


conn.row_factory = summary_factory
for summary in conn.execute(SUMMARY):
    print(summary, "spread:", summary.spread)


StationSummary(station='Svalbard', readings=8736, coldest=-17.3, warmest=8.3) spread: 25.6
StationSummary(station='Kirkenes', readings=0, coldest=None, warmest=None) spread: None
StationSummary(station='Tromso', readings=8760, coldest=-9.2, warmest=16.2) spread: 25.4
StationSummary(station='Bergen', readings=8760, coldest=-4.8, warmest=20.8) spread: 25.6
StationSummary(station='Oslo', readings=8760, coldest=-6.3, warmest=19.3) spread: 25.6


A dataclass prints its fields, lets an editor complete `summary.station`, and carries behavior such
as `spread`. A column with no field of that name, or a field with no column, raises `TypeError` when
the factory calls the class, as one of the Common errors shows, so the query and the class have to
agree. The **Dataclasses** notebook
in the **Object-Oriented Python** guide covers the class itself, and `asdict` turns an instance back
into a dictionary when JSON needs one.

### A factory for a connection, or for one cursor

A cursor takes the connection's factory at the moment it is made, and keeps it. A factory can also
be set on one cursor, which leaves the connection alone:


In [7]:
conn.row_factory = sqlite3.Row
made_before = conn.cursor()

conn.row_factory = None
own = conn.cursor()
own.row_factory = dict_factory

query = "SELECT name FROM stations WHERE name = 'Oslo'"
print("cursor made before the change:", type(made_before.execute(query).fetchone()).__name__)
print("conn.execute after the change:", type(conn.execute(query).fetchone()).__name__)
print("cursor with its own factory:  ", type(own.execute(query).fetchone()).__name__)
print("the connection's factory:     ", conn.row_factory)


cursor made before the change: Row
conn.execute after the change: tuple
cursor with its own factory:   dict
the connection's factory:      None


`made_before` still makes `Row` objects, because it copied the connection's factory when it was
created. `conn.execute` makes a new cursor on every call, so it follows the connection, now back to
tuples, and `own` makes dictionaries without changing the connection. The Python documentation
recommends setting the factory on the connection, so that every cursor agrees, which leaves a
cursor's own factory for the query that needs a different shape.

### A tuple, a Row, a dictionary or a dataclass

Every kind of row in this notebook suits a different job:

| Write | When | Why |
|---|---|---|
| no factory, a tuple | rows unpacked right beside their query, or passed straight on as parameters | nothing is built, and unpacking names the values |
| `sqlite3.Row` | most code, reading rows by column name | names and positions both work, at close to the cost of a tuple |
| a dictionary factory | rows that mostly leave as JSON, or that code changes or reads with `get` | a plain dictionary goes anywhere a dictionary does |
| a dataclass factory | rows that are things in the program, with behavior of their own | named fields, a readable `repr`, and methods |

The default is `sqlite3.Row`, set on the connection, with `dict(row)` at the one place a dictionary
is needed, such as a JSON response. A dictionary factory earns its place when most rows leave as
JSON, and a dataclass factory when the program already has a class for what a row describes, on the
cursor that fetches those rows.

### A report for the program, and JSON for a client

The pieces of this notebook in one job. The connection's factory is `sqlite3.Row`, the default for
what the program reads. `station_summaries` runs its query on a cursor of its own with
`summary_factory`, so the program receives `StationSummary` objects whatever the connection's factory
is. `frost_hours` reads its count by name from a `Row`. The same summaries then leave as JSON,
through `asdict`, with the frost hours added:


In [8]:
def station_summaries(conn):
    """Every station's summary as a StationSummary, on a cursor with its own factory."""
    cursor = conn.cursor()
    cursor.row_factory = summary_factory
    return cursor.execute(SUMMARY).fetchall()


def frost_hours(conn, station):
    """A station's hours below freezing, read by name from an sqlite3.Row."""
    row = conn.execute("""
        SELECT COUNT(*) AS hours
        FROM readings AS r JOIN stations AS s ON s.id = r.station_id
        WHERE s.name = ? AND r.celsius < 0
    """, (station,)).fetchone()
    return row["hours"]


conn.row_factory = sqlite3.Row
summaries = station_summaries(conn)

for summary in summaries:
    spread = "no readings" if summary.spread is None else f"{summary.spread} degrees apart"
    frost = frost_hours(conn, summary.station)
    print(f"{summary.station:<9} {summary.readings:>5} readings, {spread}, {frost} hours of frost")

response = [asdict(summary) | {"frost_hours": frost_hours(conn, summary.station)} for summary in summaries[:2]]
print(json.dumps(response, indent=2))


Svalbard   8736 readings, 25.6 degrees apart, 5871 hours of frost
Kirkenes      0 readings, no readings, 0 hours of frost
Tromso     8760 readings, 25.4 degrees apart, 3203 hours of frost
Bergen     8760 readings, 25.6 degrees apart, 1220 hours of frost
Oslo       8760 readings, 25.6 degrees apart, 1848 hours of frost
[
  {
    "station": "Svalbard",
    "readings": 8736,
    "coldest": -17.3,
    "warmest": 8.3,
    "frost_hours": 5871
  },
  {
    "station": "Kirkenes",
    "readings": 0,
    "coldest": null,
    "warmest": null,
    "frost_hours": 0
  }
]


The report read `summary.spread` and `summary.readings` from objects, and every frost count by name,
while the connection stayed on `sqlite3.Row` throughout. The JSON came from the same objects, so
the client's names are the query's `AS` names, and Kirkenes, with no readings, sent `null` where
the others sent numbers.

### Where each part came from

| In the report | What it relies on | The section that showed it |
|---|---|---|
| `conn.row_factory = sqlite3.Row` | rows that answer to a column's name | sqlite3.Row |
| `row["hours"]` in `frost_hours` | a name given by `AS` in the query | A dictionary factory, built from description |
| `cursor.row_factory = summary_factory` | a factory for one cursor, leaving the connection's alone | A factory for a connection, or for one cursor |
| `StationSummary` built by `summary_factory` | columns matched to fields by name | A dataclass factory |
| `summary.spread` | a method on the row's own class | A dataclass factory |
| `asdict(summary)` before `json.dumps` | JSON writes dictionaries, so a row becomes one first | What a Row is not |


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlite3-deep-dive/07-row-factories-solutions.ipynb).

**1.** With `sqlite3.Row` as the connection's factory, print Tromso's coldest reading and the hour
it was taken, reading both from the row by name.


In [9]:
# your code here


**2.** Run `SELECT name AS Station, latitude AS Lat FROM stations ORDER BY latitude DESC`, print the
first row's `keys()`, and read its latitude as `row["lat"]`.


In [10]:
# your code here


**3.** Write `rows_as_json(conn, sql, parameters=())`, which runs a query on a cursor with its own
dictionary factory and returns the rows as a JSON string. Print it for the stations north of 69
degrees, from north to south.


In [11]:
# your code here


**4.** Write a dataclass `Reading` with the fields `station`, `hour` and `celsius`, and a factory for
it, and print Oslo's three warmest readings as `Reading` objects.


In [12]:
# your code here


**5.** Write a row factory that returns only the first value of a row, and use it on one cursor to
get a plain list of every station's name, in alphabetical order.


In [13]:
# your code here


**6.** Write every station, as a dictionary made with `dict(row)`, to `scratch/stations.json`, then
read the file back with `json.load` and print the name of every station in it.


In [14]:
# your code here


## Common errors

### TypeError: tuple indices must be integers or slices, not str


In [15]:
conn.row_factory = None
row = conn.execute("SELECT name, latitude FROM stations WHERE name = ?", ("Oslo",)).fetchone()
print(row["latitude"])


TypeError: tuple indices must be integers or slices, not str

With no row factory, a row is a tuple, and a tuple answers only to positions. Set a factory that
answers to names before the query runs:


In [16]:
conn.row_factory = sqlite3.Row
row = conn.execute("SELECT name, latitude FROM stations WHERE name = ?", ("Oslo",)).fetchone()

print(row["latitude"])


59.91


### AttributeError: 'sqlite3.Row' object has no attribute 'get'


In [17]:
row = conn.execute("""
    SELECT s.name, MIN(r.celsius) AS coldest
    FROM stations AS s LEFT JOIN readings AS r ON r.station_id = s.id
    WHERE s.name = ?
""", ("Kirkenes",)).fetchone()
print(row.get("coldest", "no readings"))


AttributeError: 'sqlite3.Row' object has no attribute 'get'

`get` is a dictionary's method, and a `Row` is not a dictionary. When a column is always in the
query, as `coldest` is here, `row["coldest"]` is the lookup. `get` would not have helped anyway:
the column is there and its value is `None`, and a default only stands in for a missing key, as the
dictionary on the second line shows. Test the value itself:


In [18]:
coldest = row["coldest"]
print(coldest if coldest is not None else "no readings")

print(dict(row).get("coldest", "no readings"))


no readings
None


### IndexError: No item with that key


In [19]:
row = conn.execute("SELECT name, latitude AS lat FROM stations WHERE name = ?", ("Tromso",)).fetchone()
print(row["latitude"])


IndexError: No item with that key

The query renamed `latitude` to `lat`, and a row answers only to the names its query gave. The error
is an `IndexError`, not the `KeyError` a dictionary raises, since a `Row` is a sequence, so an
`except KeyError` written for dictionaries does not catch it, while `except LookupError` catches
both. `keys()` shows the names the row has:


In [20]:
print(row.keys())
print(row["lat"])


['name', 'lat']
69.65


### TypeError: Object of type Row is not JSON serializable


In [21]:
rows = conn.execute("SELECT name, latitude FROM stations ORDER BY latitude DESC LIMIT 2").fetchall()
json.dumps(rows)


TypeError: Object of type Row is not JSON serializable

`json` writes dictionaries, lists, strings, numbers, `True`, `False` and `None`, and a `Row` is none
of those. Turn every row into a dictionary first, or fetch dictionaries in the first place with a
dictionary factory:


In [22]:
print(json.dumps([dict(row) for row in rows]))


[{"name": "Svalbard", "latitude": 78.22}, {"name": "Kirkenes", "latitude": 69.73}]


### TypeError: StationSummary.__init__() got an unexpected keyword argument 'mean'


In [23]:
conn.row_factory = summary_factory
conn.execute("""
    SELECT s.name AS station, COUNT(r.celsius) AS readings, MIN(r.celsius) AS coldest,
           MAX(r.celsius) AS warmest, ROUND(AVG(r.celsius), 1) AS mean
    FROM stations AS s LEFT JOIN readings AS r ON r.station_id = s.id
    GROUP BY s.id
""").fetchall()


TypeError: StationSummary.__init__() got an unexpected keyword argument 'mean'

The query grew a column, and `summary_factory` passed it to a class that has no field of that name.
A field the query does not fill raises the same way, as a missing argument. The query and the class
have to agree, so add the field to the class, or name the columns the class has:


In [24]:
summaries = conn.execute(SUMMARY).fetchall()
conn.row_factory = sqlite3.Row
print(summaries[0], "spread:", summaries[0].spread)


StationSummary(station='Svalbard', readings=8736, coldest=-17.3, warmest=8.3) spread: 25.6


### No error, and null in the JSON: a column named Coldest, read as coldest


In [25]:
row = conn.execute("""
    SELECT s.name AS Station, MIN(r.celsius) AS Coldest
    FROM stations AS s JOIN readings AS r ON r.station_id = s.id
    WHERE s.name = ?
""", ("Svalbard",)).fetchone()
print("in Python: ", row["station"], row["coldest"])

received = json.loads(json.dumps(dict(row)))   # what a client of the JSON receives
print("the client:", received.get("station"), received.get("coldest"))


in Python:  Svalbard -17.3
the client: None None


A `Row` ignores case when it looks up a name, so `row["coldest"]` found `Coldest`, and the Python
side worked. A dictionary does not ignore case: `dict(row)` kept the names exactly as the query wrote
them, so the JSON said `Station` and `Coldest`, and a client asking for `station` and `coldest` got
`None` for both, with no error anywhere. Write the names in the query in the case every reader will
use:


In [26]:
row = conn.execute("""
    SELECT s.name AS station, MIN(r.celsius) AS coldest
    FROM stations AS s JOIN readings AS r ON r.station_id = s.id
    WHERE s.name = ?
""", ("Svalbard",)).fetchone()
received = json.loads(json.dumps(dict(row)))

print("the client:", received.get("station"), received.get("coldest"))


the client: Svalbard -17.3


### No error, and the station's id where the reading's belongs: two columns named id


In [27]:
coldest_reading = """
    SELECT *
    FROM stations AS s JOIN readings AS r ON r.station_id = s.id
    WHERE r.celsius IS NOT NULL
    ORDER BY r.celsius
    LIMIT 1
"""
row = conn.execute(coldest_reading).fetchone()
print(row.keys())
print("the coldest reading's id, from a Row:        ", row["id"])

cursor = conn.cursor()
cursor.row_factory = dict_factory
print("the coldest reading's id, from dict_factory: ", cursor.execute(coldest_reading).fetchone()["id"])


['id', 'name', 'latitude', 'id', 'station_id', 'hour', 'celsius']
the coldest reading's id, from a Row:         3
the coldest reading's id, from dict_factory:  1071


`SELECT *` over a join returns every column of both tables, and both tables have a column called
`id`. A `Row` answers a repeated name with the first column of that name, the station's id, while
`dict_factory` wrote the key `id` twice and kept the second value, the reading's. Neither raised, and
they disagree. Name the columns a join returns, and give a repeated name a name of its own:


In [28]:
row = conn.execute("""
    SELECT s.id AS station_id, r.id AS reading_id, s.name, r.hour, r.celsius
    FROM stations AS s JOIN readings AS r ON r.station_id = s.id
    WHERE r.celsius IS NOT NULL
    ORDER BY r.celsius
    LIMIT 1
""").fetchone()

print(dict(row))
conn.close()


{'station_id': 3, 'reading_id': 1071, 'name': 'Svalbard', 'hour': '2025-01-12T03:00', 'celsius': -17.3}


Last, the notebook is finished with its files, so this cell removes the scratch folder, with the
database in it:


In [29]:
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


## Recap

- A row factory turns every row's tuple of values into what `fetchone`, `fetchall` and a loop return,
  and the default, `None`, keeps the tuple.
- `sqlite3.Row` answers to a column's name, in any case, and to its position, lists its names with
  `keys()`, and becomes a dictionary with `dict(row)`.
- A `Row` is a sequence, not a dictionary: it has no `get`, `in` looks at its values, a name it lacks
  raises `IndexError`, and `json` needs `dict(row)`.
- A factory of your own takes the cursor and the tuple and builds names from `description`: a
  dictionary for rows that leave as JSON, or a dataclass for rows with behavior of their own.
- The query decides the names: `AS` sets them, in the case every reader will use, and a join names
  its columns so that no name repeats.
- `conn.row_factory` reaches the cursors made after it, `cursor.row_factory` one cursor, and
  `sqlite3.Row` on the connection is the default to reach for.


## What is next

The **Type Affinity** notebook looks at the values inside the rows: why a column declared `INTEGER`
accepts text, what SQLite does with the type a value arrives with, and what a `STRICT` table changes.


---

&#8592; **Previous:** [Parameters](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlite3-deep-dive/06-parameters.ipynb)  &nbsp;·&nbsp;  [sqlite3, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlite3-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [Type Affinity](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlite3-deep-dive/08-type-affinity.ipynb) &#8594;
